# 📊 01. Exploratory Data Analysis & Imbalance Characterization
### Project: XAI System in Banking: AI Anomaly-Driven Credit Card Fraud Detection and Interpretation
**Objective:** In-depth exploratory data analysis of the European cardholder credit card fraud dataset (`mlg-ulb/creditcardfraud`), characterizing extreme class imbalance ($0.172\%$), feature distributions, time-domain dynamics, and financial transaction amounts.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add repo root to path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

# Style settings
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.dpi"] = 120
print(f"Working directory: {ROOT_DIR}")

## 1. Data Ingestion & Shape Verification
The primary dataset consists of credit card transactions made by European cardholders in September 2013 over a two-day period.

In [ ]:
data_path = ROOT_DIR / "data/raw/creditcard.csv"
if not data_path.exists():
    from scripts.download_data import download_dataset
    download_dataset("mlg-ulb/creditcardfraud", str(ROOT_DIR / "data/raw"))

df = pd.read_csv(data_path)
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

## 2. Extreme Class Imbalance Characterization
In credit card fraud detection, fraudulent transactions represent a minute fraction of overall transaction volume. We evaluate the exact class distribution below.

In [ ]:
class_counts = df["Class"].value_counts()
normal_count = class_counts[0]
fraud_count = class_counts[1]
fraud_pct = (fraud_count / len(df)) * 100

print(f"Legitimate Transactions (Class 0): {normal_count:,} ({100 - fraud_pct:.4f}%)")
print(f"Fraudulent Transactions (Class 1): {fraud_count:,} ({fraud_pct:.4f}%)")
print(f"Imbalance Ratio: ~{normal_count // fraud_count}:1")

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=["Legitimate (0)", "Fraud (1)"], y=[normal_count, fraud_count], palette=["#2b5c8f", "#d9534f"], ax=ax)
ax.set_yscale("log")
ax.set_title("Transaction Class Distribution (Logarithmic Scale)", fontsize=12, fontweight="bold")
ax.set_ylabel("Count (log scale)")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha="center", va="center", xytext=(0, 6), textcoords="offset points", fontsize=10)
plt.tight_layout()
plt.show()

## 3. Transaction Amount Analysis & Logarithmic Transformation
Transaction amounts exhibit extreme right-skewness: while legitimate transactions span small everyday amounts up to corporate expenses, fraud patterns often concentrate in distinct amounts to avoid trigger thresholds.

In [ ]:
print("Transaction Amount Summary:")
print("Legitimate:", df[df["Class"] == 0]["Amount"].describe())
print("\nFraudulent:", df[df["Class"] == 1]["Amount"].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[df["Class"] == 0]["Amount"], bins=50, kde=True, color="#2b5c8f", ax=axes[0])
axes[0].set_title("Legitimate Amount Distribution ($)", fontsize=11, fontweight="bold")
axes[0].set_xlim([0, 1500])

sns.histplot(df[df["Class"] == 1]["Amount"], bins=50, kde=True, color="#d9534f", ax=axes[1])
axes[1].set_title("Fraudulent Amount Distribution ($)", fontsize=11, fontweight="bold")
axes[1].set_xlim([0, 1500])
plt.tight_layout()
plt.show()

## 4. Temporal Dynamics over 48 Hours
The `Time` feature records elapsed seconds from the first recorded transaction over 48 hours. By extracting the hour of the day (`(Time // 3600) % 24`), we observe distinct circadian patterns in fraudulent activity.

In [ ]:
df["Hour"] = (df["Time"] // 3600) % 24

plt.figure(figsize=(10, 4))
sns.kdeplot(df[df["Class"] == 0]["Hour"], label="Legitimate", color="#2b5c8f", lw=2)
sns.kdeplot(df[df["Class"] == 1]["Hour"], label="Fraud", color="#d9534f", lw=2)
plt.title("Diurnal Transaction Density: Legitimate vs Fraud (Hour of Day)", fontsize=12, fontweight="bold")
plt.xlabel("Hour of Day (0 - 23)")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()

## 5. PCA Component Correlation Matrix
Features `V1` through `V28` are orthogonal PCA components resulting from confidential dimensionality reduction. We inspect correlations with the target class.

In [ ]:
corr = df.corr()["Class"].drop(["Class", "Hour"]).sort_values()

plt.figure(figsize=(10, 5))
colors = ["#d9534f" if v < -0.15 or v > 0.15 else "#4a90e2" for v in corr.values]
corr.plot(kind="bar", color=colors)
plt.title("Feature Correlation with Fraud Label ('Class')", fontsize=12, fontweight="bold")
plt.ylabel("Pearson Correlation Coefficient")
plt.axhline(0, color="black", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

print("Top 5 Negative Correlations (Lower value -> Higher Fraud Risk):")
print(corr.head(5))
print("\nTop 5 Positive Correlations (Higher value -> Higher Fraud Risk):")
print(corr.tail(5)[::-1])